# 01 — Data Preparation & Exploratory Data Analysis

**Project:** Supply Chain Analysis and Prediction  
**Dataset:** Kaggle — discovertalent143/supply-chain-dataset  

## Purpose
Understand the raw data before any transformation — distributions, missing values, correlations, and target balance.

## Flow

| Step | Description |
|------|-------------|
| 1 | Data Overview — shape, dtypes, missing values, duplicates |
| 2 | Missing Value Analysis |
| 3 | Univariate Analysis — numeric & categorical distributions |
| 4 | Bivariate / Correlation Analysis |
| 5 | Target Variable Analysis |
| 6 | Key Findings Summary |

---

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.data_loader import load_raw_data, quick_summary
from src.config import IMAGES_DIR, PLOT_STYLE, FIGURE_DPI

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use(PLOT_STYLE)
pd.set_option('display.max_columns', None)
print('Setup complete.')

Setup complete.


## 1. Data Overview

In [2]:
df = load_raw_data()
print(f'Shape: {df.shape}')
display(df.head())

[data_loader] Loaded 50 rows x 10 columns
[data_loader] Columns: ['Product', 'Supplier', 'Warehouse_Location', 'Quantity', 'Unit_Price', 'Total_Cost', 'Delivery_Date', 'Logistics_Partner', 'Shipping_Method', 'Delivery_Status']
Shape: (50, 10)


,Product,Supplier,Warehouse_Location,Quantity,Unit_Price,Total_Cost,Delivery_Date,Logistics_Partner,Shipping_Method,Delivery_Status
0,Widget A,Alpha Corp,Warehouse 3,45,16.02,720.90,2025-06-05,FastTrans,Air,Pending
1,Widget A,Alpha Corp,Warehouse 1,37,15.47,572.39,2025-06-20,FastTrans,Road,Pending
2,Widget B,Delta LLC,Warehouse 3,45,41.42,1863.90,2025-06-01,ShipSmart,Sea,Delayed
3,Gadget X,Beta Ltd,Warehouse 1,53,9.60,508.80,2025-06-13,FastTrans,Rail,Delayed
4,Tool Z,Gamma Inc,Warehouse 1,68,29.13,1980.84,2025-06-30,TransEdge,Air,Delayed


In [3]:
print('--- Data Types ---')
display(df.dtypes.to_frame('dtype'))

print('\n--- Descriptive Statistics (Numeric) ---')
display(df.describe())

--- Data Types ---


,dtype
Product,str
Supplier,str
Warehouse_Location,str
Quantity,int64
Unit_Price,float64
Total_Cost,float64
Delivery_Date,str
Logistics_Partner,str
Shipping_Method,str
Delivery_Status,str



--- Descriptive Statistics (Numeric) ---


,Quantity,Unit_Price,Total_Cost
count,50.000000,50.000000,50.000000
mean,50.340000,28.227800,1450.923200
std,23.556801,14.744061,1181.068132
min,10.000000,5.950000,146.400000
25%,34.250000,13.560000,584.497500
50%,45.000000,28.190000,1038.585000
75%,68.000000,43.760000,1945.157500
max,93.000000,49.930000,4583.970000


In [4]:
print(f'Duplicate rows : {df.duplicated().sum()}')
print(f'Total rows     : {len(df)}')
print(f'Total columns  : {df.shape[1]}')
print('\nMissing values per column:')
display(df.isnull().sum().to_frame('missing_count'))

Duplicate rows : 0
Total rows     : 50
Total columns  : 10

Missing values per column:


,missing_count
Product,0
Supplier,0
Warehouse_Location,0
Quantity,0
Unit_Price,0
Total_Cost,0
Delivery_Date,0
Logistics_Partner,0
Shipping_Method,0
Delivery_Status,0


## 2. Missing Value Analysis

In [5]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print('No missing values found in the dataset.')
else:
    display(missing_df)
    fig, ax = plt.subplots(figsize=(10, 5))
    missing_df['Missing Count'].plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Missing Values per Column')
    ax.set_ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    fig.savefig(IMAGES_DIR / '01_missing_values.png', dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()
    print(f'Plot saved -> images/01_missing_values.png')

No missing values found in the dataset.


## 3. Univariate Analysis

In [6]:
# ── Numeric distributions ──
num_cols = df.select_dtypes(include='number').columns.tolist()
print(f'Numeric columns: {num_cols}')

n = len(num_cols)
cols_per_row = 3
rows = (n + cols_per_row - 1) // cols_per_row

fig, axes = plt.subplots(rows, cols_per_row, figsize=(cols_per_row * 5, rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=20, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    skew = df[col].skew()
    axes[i].annotate(f'Skew: {skew:.2f}', xy=(0.7, 0.9), xycoords='axes fraction', fontsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Numeric Feature Distributions', fontsize=14)
plt.tight_layout()
fig.savefig(IMAGES_DIR / '02_numeric_distributions.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/02_numeric_distributions.png')

Numeric columns: ['Quantity', 'Unit_Price', 'Total_Cost']


Plot saved -> images/02_numeric_distributions.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10932\1476258740.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ── Categorical distributions ──
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns: {cat_cols}')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols[:6]):
    counts = df[col].value_counts()
    counts.plot(kind='bar', ax=axes[i], color='teal', edgecolor='white')
    axes[i].set_title(f'{col} — Value Counts')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Count')
    plt.setp(axes[i].get_xticklabels(), rotation=30, ha='right')

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Categorical Feature Distributions', fontsize=14)
plt.tight_layout()
fig.savefig(IMAGES_DIR / '02b_categorical_distributions.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/02b_categorical_distributions.png')

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10932\474571786.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


Categorical columns: ['Product', 'Supplier', 'Warehouse_Location', 'Delivery_Date', 'Logistics_Partner', 'Shipping_Method', 'Delivery_Status']


Plot saved -> images/02b_categorical_distributions.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10932\474571786.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Bivariate & Correlation Analysis

In [8]:
# ── Correlation heatmap ──
corr = df.select_dtypes(include='number').corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
fig.savefig(IMAGES_DIR / '03_correlation_heatmap.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/03_correlation_heatmap.png')

print('\nCorrelation matrix:')
display(corr.round(3))

Plot saved -> images/03_correlation_heatmap.png

Correlation matrix:


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10932\558559265.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Quantity,Unit_Price,Total_Cost
Quantity,1.000,0.088,0.668
Unit_Price,0.088,1.000,0.727
Total_Cost,0.668,0.727,1.000


In [9]:
# ── Scatter: Quantity vs Total Cost colored by Delivery Status ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Quantity vs Total Cost
for status, grp in df.groupby('Delivery_Status'):
    axes[0].scatter(grp['Quantity'], grp['Total_Cost'], label=status, alpha=0.7)
axes[0].set_xlabel('Quantity')
axes[0].set_ylabel('Total Cost')
axes[0].set_title('Quantity vs Total Cost by Delivery Status')
axes[0].legend()

# Unit Price vs Total Cost
for status, grp in df.groupby('Delivery_Status'):
    axes[1].scatter(grp['Unit_Price'], grp['Total_Cost'], label=status, alpha=0.7)
axes[1].set_xlabel('Unit Price')
axes[1].set_ylabel('Total Cost')
axes[1].set_title('Unit Price vs Total Cost by Delivery Status')
axes[1].legend()

plt.tight_layout()
fig.savefig(IMAGES_DIR / '03b_scatter_bivariate.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/03b_scatter_bivariate.png')

Plot saved -> images/03b_scatter_bivariate.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10932\3928492957.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# ── Delivery Status by Shipping Method ──
ct = pd.crosstab(df['Shipping_Method'], df['Delivery_Status'])
print('Delivery Status by Shipping Method:')
display(ct)

fig, ax = plt.subplots(figsize=(10, 5))
ct.plot(kind='bar', ax=ax, edgecolor='white')
ax.set_title('Delivery Status by Shipping Method')
ax.set_xlabel('Shipping Method')
ax.set_ylabel('Count')
plt.xticks(rotation=0)
ax.legend(title='Delivery Status')
plt.tight_layout()
fig.savefig(IMAGES_DIR / '03c_status_by_shipping.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/03c_status_by_shipping.png')

Delivery Status by Shipping Method:


Delivery_Status,Delayed,Delivered,In Transit,Pending
Shipping_Method,,,,
Air,4,1,3,3
Rail,2,2,3,3
Road,6,5,4,3
Sea,4,2,4,1


Plot saved -> images/03c_status_by_shipping.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10932\1047292098.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# ── Avg Total Cost by Supplier ──
fig, ax = plt.subplots(figsize=(10, 5))
df.groupby('Supplier')['Total_Cost'].mean().sort_values(ascending=False).plot(
    kind='bar', ax=ax, color='darkorange', edgecolor='white'
)
ax.set_title('Average Total Cost by Supplier')
ax.set_ylabel('Avg Total Cost ($)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
fig.savefig(IMAGES_DIR / '03d_cost_by_supplier.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/03d_cost_by_supplier.png')

Plot saved -> images/03d_cost_by_supplier.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10932\3550358048.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Target Variable Analysis

In [12]:
# ── Delivery Status distribution ──
status_counts = df['Delivery_Status'].value_counts()
print('Delivery Status distribution:')
display(status_counts.to_frame('count'))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

status_counts.plot(kind='bar', ax=axes[0], color=['tomato','steelblue','mediumseagreen','darkorange'],
                   edgecolor='white')
axes[0].set_title('Delivery Status Counts')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right')
for p in axes[0].patches:
    axes[0].annotate(str(int(p.get_height())),
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom')

status_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90,
                   colors=['tomato','steelblue','mediumseagreen','darkorange'])
axes[1].set_title('Delivery Status Share')
axes[1].set_ylabel('')

plt.tight_layout()
fig.savefig(IMAGES_DIR / '04_target_distribution.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/04_target_distribution.png')

Delivery Status distribution:


,count
Delivery_Status,
Delayed,16
In Transit,14
Pending,10
Delivered,10


Plot saved -> images/04_target_distribution.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10932\1430055414.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# ── Binary Delay Label ──
df['Delay_Label'] = (df['Delivery_Status'] == 'Delayed').astype(int)
delay_rate = df['Delay_Label'].mean() * 100
print(f'Delay rate: {delay_rate:.1f}% ({df["Delay_Label"].sum()} out of {len(df)} orders)')

# Delay rate by shipping method
delay_by_ship = df.groupby('Shipping_Method')['Delay_Label'].mean().sort_values(ascending=False)
print('\nDelay rate by Shipping Method:')
display(delay_by_ship.to_frame('Delay Rate').style.format('{:.1%}'))

# Delay rate by warehouse
delay_by_wh = df.groupby('Warehouse_Location')['Delay_Label'].mean().sort_values(ascending=False)
print('\nDelay rate by Warehouse:')
display(delay_by_wh.to_frame('Delay Rate').style.format('{:.1%}'))

Delay rate: 32.0% (16 out of 50 orders)



Delay rate by Shipping Method:


,Delay Rate
Shipping_Method,
Air,36.4%
Sea,36.4%
Road,33.3%
Rail,20.0%



Delay rate by Warehouse:


,Delay Rate
Warehouse_Location,
Warehouse 2,46.7%
Warehouse 3,28.6%
Warehouse 1,23.8%


## 6. Key Findings Summary

| # | Finding |
|---|--------|
| 1 | Dataset has **50 orders**, **10 columns**, **0 missing values**, **0 duplicates** — clean dataset |
| 2 | **32% of orders are delayed** (16/50) — class imbalance is mild |
| 3 | **Total_Cost** is strongly correlated with **Quantity** (as expected — more units = higher cost) |
| 4 | **Unit_Price** ranges widely ($5.95–$49.93) — high variance in product pricing |
| 5 | **Road** is the most used shipping method (36%); delay rates vary by shipping mode |
| 6 | **Warehouse 1** handles the most orders (42%); efficiency varies by location |
| 7 | **Alpha Corp** is the top supplier by order volume |
| 8 | No missing values means no imputation needed — proceed directly to feature engineering |

---

**Next step:** `02_Feature_Engineering.ipynb` — create derived features and encode categoricals for modelling.